In [2]:
from __future__ import annotations
import math, random, uuid
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from enum import Enum
import numpy as np
import matplotlib, matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
import seaborn as sns
from scipy.stats import norm as scipy_norm
import plotly.graph_objects as go
import plotly.colors as pc
import plotly.io as pio
from scipy.interpolate import RegularGridInterpolator
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
pio.renderers.default = "notebook_connected"
matplotlib.rcParams.update({"figure.dpi":120,"axes.spines.top":False,"axes.spines.right":False})
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic=True
seed_everything(42); print("Ready.")


## E2. FTCS Simulator

@dataclass
class RodConfigSrc:
    L:float=0.1; D:float=4.25e-6; N:int=100; dt:float=1e-3; t_end:float=100.
    T_left:float=90.; T_right:float=90.; T_init:float=20.
    source_locs:tuple=(); source_temps:tuple=()

def simulate_heat_ftcs_src(cfg, save_times):
    N=cfg.N; x=np.linspace(0.,cfg.L,N+1); dx=cfg.L/N; c=cfg.D*cfg.dt/dx**2
    if c>.5: raise ValueError(f"Unstable c={c:.4g}")
    T=np.full(N+1,cfg.T_init,dtype=np.float64); T[0]=cfg.T_left; T[-1]=cfg.T_right; Tnew=T.copy()
    src_idx=[int(np.clip(round(sl/dx),1,N-1)) for sl in cfg.source_locs]; src_t=list(cfg.source_temps)
    for i,st in zip(src_idx,src_t): T[i]=st
    sv=np.clip(np.asarray(save_times,np.float64),0.,cfg.t_end); ss=np.rint(sv/cfg.dt).astype(int)
    ms=int(round(cfg.t_end/cfg.dt)); Thist=np.empty((len(sv),N+1),np.float64); ptr=0
    for step in range(ms+1):
        while ptr<len(ss) and step==ss[ptr]: Thist[ptr]=T; ptr+=1
        if step==ms: break
        Tnew[1:-1]=T[1:-1]+c*(T[2:]+T[:-2]-2.*T[1:-1]); Tnew[0]=cfg.T_left; Tnew[-1]=cfg.T_right
        for i,st in zip(src_idx,src_t): Tnew[i]=st
        T,Tnew=Tnew,T
    print(f"  FTCS c={c:.4g}  Nx={len(x)}  Nt={len(sv)}  sources={len(src_idx)}")
    return x, sv, Thist, c


## E3. Dataset Builder & Split

def _bd(x,t,Thist,time_idx,x_idx):
    tt=t[time_idx]; xx=x[x_idx]; sub=Thist[np.ix_(time_idx,x_idx)]
    Xx,Xt=np.meshgrid(xx,tt); return np.stack([Xx.ravel(),Xt.ravel()],1), sub.ravel()

def split_random_frac(x,t,Thist,train_frac=.65,val_frac=.15,heldout_x_frac=.10,seed=42):
    rng=np.random.default_rng(seed); Nt,Nx=len(t),len(x); idx=np.arange(Nt); rng.shuffle(idx)
    n_tr=int(round(train_frac*Nt)); n_vu=int(round(val_frac*Nt))
    tr_t=np.sort(idx[:n_tr]); vu_t=np.sort(idx[n_tr:n_tr+n_vu]); te_t=np.sort(idx[n_tr+n_vu:])
    xi=np.arange(Nx); rng.shuffle(xi); n_h=max(1,int(round(heldout_x_frac*Nx)))
    hx=np.sort(xi[:n_h]); tx=np.sort(xi[n_h:]); ax=np.arange(Nx)
    data=dict(train=_bd(x,t,Thist,tr_t,tx),val_seen=_bd(x,t,Thist,tr_t,hx),
              val_unseen=_bd(x,t,Thist,vu_t,ax),test_unseen=_bd(x,t,Thist,te_t,ax))
    for k,v in data.items(): print(f"  {k:15s} {len(v[1]):7,} pts")
    return dict(idx=dict(train_t=tr_t,val_unseen_t=vu_t,test_unseen_t=te_t,train_x=tx,heldout_x=hx),data=data)

def regression_metrics(yt,yp):
    return dict(RMSE=math.sqrt(mean_squared_error(yt,yp)),MAE=mean_absolute_error(yt,yp),R2=r2_score(yt,yp))


## E4. MLP Surrogate

class MLP(nn.Module):
    def __init__(self,hidden=128,depth=4,dropout=0.):
        super().__init__(); layers=[]; dim=2
        for _ in range(depth):
            layers+=[nn.Linear(dim,hidden),nn.ReLU()]
            if dropout>0: layers.append(nn.Dropout(dropout))
            dim=hidden
        layers.append(nn.Linear(dim,1)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

def train_mlp(Xtr,ytr,Xvs,yvs,Xvu,yvu,*,epochs=3000,batch=2048,lr=1e-3,wd=1e-6,
              hidden=128,depth=4,patience=300,device=None):
    if device is None: device="cuda" if torch.cuda.is_available() else "cpu"
    model=MLP(hidden,depth).to(device)
    opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=wd); loss_fn=nn.MSELoss()
    loader=DataLoader(TensorDataset(torch.from_numpy(Xtr).float(),
                      torch.from_numpy(ytr).float().unsqueeze(1)),batch_size=batch,shuffle=True)
    def tt(X): return torch.from_numpy(X).float().to(device)
    Xvs_t,yvs_t=tt(Xvs),torch.from_numpy(yvs).float().unsqueeze(1).to(device)
    Xvu_t,yvu_t=tt(Xvu),torch.from_numpy(yvu).float().unsqueeze(1).to(device)
    hist=dict(tr=[],vs=[],vu=[]); best,bad,best_st=float("inf"),0,None
    for ep in range(1,epochs+1):
        model.train(); tl=[]
        for xb,yb in loader:
            xb,yb=xb.to(device),yb.to(device); opt.zero_grad(set_to_none=True)
            l=loss_fn(model(xb),yb); l.backward(); opt.step(); tl.append(l.item())
        model.eval()
        with torch.no_grad(): vs=loss_fn(model(Xvs_t),yvs_t).item(); vu=loss_fn(model(Xvu_t),yvu_t).item()
        tr=float(np.mean(tl)); hist["tr"].append(tr); hist["vs"].append(vs); hist["vu"].append(vu)
        if vu<best-1e-12: best=vu; best_st={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=patience: break
        if ep%50==0 or ep==1: print(f"  ep{ep:5d}  tr={tr:.3e}  vs={vs:.3e}  vu={vu:.3e}  bad={bad}")
    if best_st: model.load_state_dict(best_st)
    return model, hist

def mlp_predict_fn(model,scaler,device=None):
    if device is None: device="cuda" if torch.cuda.is_available() else "cpu"
    def predict(X):
        with torch.no_grad(): p=model(torch.from_numpy(scaler.transform(X)).float().to(device)).detach().cpu().numpy().ravel()
        return p, None
    return predict

def plot_train_curves(hist,title):
    plt.figure(figsize=(8,3.5))
    for k,c in [("tr","C0"),("vs","C1"),("vu","C2")]: plt.plot(hist[k],label=k,color=c,lw=1.3)
    plt.yscale("log"); plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.title(title); plt.legend(); plt.tight_layout(); plt.show()


## E5. GPR Surrogate

def train_gpr(Xtr,ytr,max_pts=5000,seed=42):
    rng=np.random.default_rng(seed); n=len(Xtr)
    if n>max_pts: i=rng.choice(n,size=max_pts,replace=False); Xtr,ytr=Xtr[i],ytr[i]; print(f"  GPR ↓ {max_pts}")
    k=C(1.,(1e-2,1e3))*RBF([1.,1.],(1e-3,1e3))+WhiteKernel(1e-3,(1e-8,1.))
    gpr=GaussianProcessRegressor(kernel=k,alpha=0.,normalize_y=True,n_restarts_optimizer=2,random_state=seed)
    gpr.fit(Xtr,ytr); print(f"  GPR kernel={gpr.kernel_}"); return gpr

def gpr_predict_fn(gpr,scaler):
    def predict(X): m,s=gpr.predict(scaler.transform(X),return_std=True); return m.ravel(),s.ravel()
    return predict


## E6. Adaptive Sampling Machinery

class SamplingStrategy(Enum): UNIFORM="uniform"; GAP_WEIGHTED="gap_weighted"

@dataclass
class EnvelopeData:
    time:float; x:np.ndarray; T:np.ndarray; T_under:np.ndarray; T_over:np.ndarray
    gap:np.ndarray; max_gap:float; lower_hull_idx:np.ndarray; upper_hull_idx:np.ndarray

@dataclass
class SamplingPlan:
    
    time:float
    total_samples:int
    gap_allocation:Dict[int,int]
    sample_indices:Dict[int,np.ndarray]
    local_envelopes:Dict[int,Dict]=field(default_factory=dict)

class RodSimulationSrc:
    def __init__(self,cfg):
        self.cfg=cfg
        N=cfg.N
        dx=cfg.L/N 
        self.x=np.linspace(0.,cfg.L,N+1)
        self.c=cfg.dt*cfg.D/dx**2; assert self.c<=.5
        self.src_idx=[int(np.clip(round(sl/dx),1,N-1)) for sl in cfg.source_locs]
        
        self.src_temps=list(cfg.source_temps)
        
        self.T=np.full(N+1,cfg.T_init,dtype=float)
        self.T[0]=cfg.T_left
        self.T[N]=cfg.T_right
        
        for i,st in zip(self.src_idx,self.src_temps): 
            self.T[i]=st
        self.current_time=0.
        
    def advance_to(self,t_target):
        N=self.cfg.N
        n=max(0,int(round((t_target-self.current_time)/self.cfg.dt)))
        
        for _ in range(n):
            Tn=self.T.copy()
            Tn[1:N]=self.T[1:N]+self.c*(self.T[2:N+1]+self.T[0:N-1]-2.*self.T[1:N])
            Tn[0]=self.cfg.T_left; Tn[N]=self.cfg.T_right
            for i,st in zip(self.src_idx,self.src_temps): Tn[i]=st
            self.T=Tn
        self.current_time=t_target; return self.T.copy()

class McCormickEnvelope:
    @staticmethod
    def _lower(x,y):
        idx=[]
        for k in range(len(x)):
            idx.append(k)
            while len(idx)>=3:
                i,j,l=idx[-3],idx[-2],idx[-1]
                if (y[j]-y[i])/(x[j]-x[i])>=(y[l]-y[j])/(x[l]-x[j]): idx.pop(-2)
                else: break
        return np.array(idx,int)
    @staticmethod
    def _upper(x,y):
        idx=[]
        for k in range(len(x)):
            idx.append(k)
            while len(idx)>=3:
                i,j,l=idx[-3],idx[-2],idx[-1]
                if (y[j]-y[i])/(x[j]-x[i])<=(y[l]-y[j])/(x[l]-x[j]): idx.pop(-2)
                else: break
        return np.array(idx,int)
    @classmethod
    def compute(cls,x,T):
        if len(x)<3:
            z=np.zeros_like(T)
            return dict(T_under=T.copy(),T_over=T.copy(),gap=z,max_gap=0.,li=np.arange(len(x)),ui=np.arange(len(x)))
        li=cls._lower(x,T); ui=cls._upper(x,T); Tu=np.interp(x,x[li],T[li]); To=np.interp(x,x[ui],T[ui]); g=To-Tu
        return dict(T_under=Tu,T_over=To,gap=g,max_gap=float(np.max(g)),li=li,ui=ui)
    @classmethod
    def compute_full(cls,x,T,time)->EnvelopeData:
        d=cls.compute(x,T)
        return EnvelopeData(time=time,x=x,T=T,T_under=d["T_under"],T_over=d["T_over"],gap=d["gap"],max_gap=d["max_gap"],lower_hull_idx=d["li"],upper_hull_idx=d["ui"])

class GapAnalyzer:
    @staticmethod
    def allocate(local_max_gaps,total,min_s=2):
        N=len(local_max_gaps); min_s=min(min_s,max(1,total//(2*N)))
        tot_gap=sum(local_max_gaps.values()); floor=min_s*N; surplus=total-floor
        if tot_gap>0 and surplus>0:
            raw={k:g/tot_gap*surplus for k,g in local_max_gaps.items()}
            alloc={k:min_s+int(raw[k]) for k in raw}; rem=total-sum(alloc.values())
            for k in sorted(raw,key=lambda k:raw[k]-int(raw[k]),reverse=True)[:abs(rem)]: alloc[k]+=int(np.sign(rem))
        else:
            base=total//N; alloc={k:base for k in local_max_gaps}
            for k in list(local_max_gaps)[:total-base*N]: alloc[k]+=1
        assert sum(alloc.values())==total; return alloc

class AdaptiveSampler:
    def __init__(self,max_time,record_interval,n_intervals,seed=42):
        self.max_time=max_time; self.record_interval=record_interval; self.n_intervals=n_intervals
        self.rng=np.random.default_rng(seed); self.record_times=np.arange(0.,max_time+1e-9,record_interval)
        self.envelope_history:List[EnvelopeData]=[]; self.sampling_plans:List[SamplingPlan]=[]
    def record(self,env): self.envelope_history.append(env)
    def allocate_global_budget(self,total_budget,min_s=5):
        n=len(self.envelope_history); mc=McCormickEnvelope(); ga=GapAnalyzer()
        max_gaps=np.array([e.max_gap for e in self.envelope_history]); gap_sum=np.sum(max_gaps)
        sps={i:max(min_s,int(np.round(total_budget*g/gap_sum))) if gap_sum>1e-10
              else (total_budget//n) for i,g in enumerate(max_gaps)}
        rem=total_budget-sum(sps.values())
        for i in sorted(range(n),key=lambda k:max_gaps[k],reverse=True)[:abs(rem)]: sps[i]+=int(np.sign(rem))
        edges=np.linspace(self.envelope_history[0].x[0],self.envelope_history[0].x[-1],self.n_intervals+1)
        for si,env in enumerate(self.envelope_history):
            n_si=sps[si]; local_envs={}
            for k in range(self.n_intervals):
                mask=(env.x>=edges[k])&(env.x<=edges[k+1]); xi,Ti=env.x[mask],env.T[mask]; lv=mc.compute(xi,Ti)
                local_envs[k]=dict(x=xi,T=Ti,T_under=lv["T_under"],T_over=lv["T_over"],gap=lv["gap"],max_gap=lv["max_gap"])
            lmg={k:local_envs[k]["max_gap"] for k in range(self.n_intervals)}
            alloc=ga.allocate(lmg,n_si,min_s); sample_idx={}
            for k in range(self.n_intervals):
                if alloc[k]>0: sample_idx[k]=np.linspace(edges[k],edges[k+1],alloc[k])
            self.sampling_plans.append(SamplingPlan(time=env.time,total_samples=n_si,gap_allocation=alloc,sample_indices=sample_idx,local_envelopes=local_envs))
        print(f"  Adaptive budget={total_budget}  records={n}  done.")


## E7. Base Visualisation Helpers

def plot_field_src(x,t,Z,title,cbar_label="T (°C)",src_x=None,ax=None):
    own=ax is None
    if own: fig,ax=plt.subplots(figsize=(8,3.5))
    im=ax.imshow(Z,aspect="auto",origin="lower",extent=[x.min(),x.max(),t.min(),t.max()])
    plt.colorbar(im,ax=ax,label=cbar_label)
    if src_x:
        for sx in src_x: ax.axvline(sx,color="red",lw=1.5,ls="--",alpha=.75)
    ax.set_xlabel("x (m)"); ax.set_ylabel("t (s)"); ax.set_title(title)
    if own: plt.tight_layout(); plt.show()

def plot_pred_vs_true(yt,yp,title,ax=None):
    own=ax is None
    if own: fig,ax=plt.subplots(figsize=(5,5))
    ax.scatter(yt,yp,s=5,alpha=.35); mn,mx=min(yt.min(),yp.min()),max(yt.max(),yp.max())
    ax.plot([mn,mx],[mn,mx],"r--",lw=1); ax.set_xlabel("True T"); ax.set_ylabel("Pred T"); ax.set_title(title)
    if own: plt.tight_layout(); plt.show()

def plot_profiles_src(x,t,Thist,mlp_fn,gpr_fn,times_plot,src_x,title):
    n=len(times_plot); fig,axes=plt.subplots(1,n,figsize=(4*n,4),sharey=True)
    for ax,pt in zip(axes,times_plot):
        ki=int(np.argmin(np.abs(t-pt))); Xq=np.stack([x,np.full_like(x,t[ki])],1)
        ax.plot(x,Thist[ki],"k-",lw=2,label="Truth"); ax.plot(x,mlp_fn(Xq)[0],"b--",lw=1.5,label="MLP")
        m,s=gpr_fn(Xq); ax.plot(x,m,"g-.",lw=1.5,label="GPR"); ax.fill_between(x,m-2*s,m+2*s,alpha=.15,color="green")
        if src_x:
            for sx in src_x: ax.axvline(sx,color="red",lw=1,ls="--",alpha=.5)
        ax.set_title(f"t={t[ki]:.1f}s"); ax.set_xlabel("x (m)")
        if ax is axes[0]: ax.set_ylabel("T (°C)")
        ax.legend(fontsize=7)
    fig.suptitle(title,fontsize=11); plt.tight_layout(); plt.show()

def plot_subdomain_envelopes_src(sampler,time_value=None,time_index=0,ax=None):
    if time_value is not None:
        times=[e.time for e in sampler.envelope_history]
        time_index=int(np.argmin(np.abs(np.array(times)-time_value)))
    env=sampler.envelope_history[time_index]; plan=sampler.sampling_plans[time_index]
    try: cmap=plt.colormaps.get_cmap("tab10")
    except: cmap=cm.get_cmap("tab10",sampler.n_intervals)
    own=ax is None
    if own: _,ax=plt.subplots(figsize=(10,4))
    for i,lenv in plan.local_envelopes.items():
        col=cmap(i/sampler.n_intervals)
        ax.fill_between(lenv["x"],lenv["T_under"],lenv["T_over"],alpha=.2,color=col)
        ax.plot(lenv["x"],lenv["T_over"],"--",color=col,lw=1.1); ax.plot(lenv["x"],lenv["T_under"],":",color=col,lw=1.1)
        if i in plan.sample_indices:
            sx=plan.sample_indices[i]; sT=np.interp(sx,env.x,env.T)
            ax.scatter(sx,sT,color=col,s=18,zorder=8,edgecolors="k",lw=.3,label=f"I{i} gap={lenv['max_gap']:.1f}°C")
    ax.plot(env.x,env.T,"k-",lw=2,zorder=9,label="T(x)")
    ax.set_xlabel("x (m)"); ax.set_ylabel("T (°C)"); ax.set_title(f"Envelopes  t={env.time:.1f}s"); ax.legend(fontsize=7,ncol=2); ax.grid(alpha=.3)
    if own: plt.tight_layout(); plt.show()

def gap_vs_allocation_mpl(sampler,title):
    plans=sampler.sampling_plans; n=sampler.n_intervals; times=[p.time for p in plans]
    T_arr=np.array([[plans[ti].local_envelopes[k]["max_gap"] for ti in range(len(plans))] for k in range(n)])
    A_arr=np.array([[plans[ti].gap_allocation[k]             for ti in range(len(plans))] for k in range(n)])
    try: cmap=plt.colormaps.get_cmap("tab10")
    except: cmap=cm.get_cmap("tab10",n)
    fig,axes=plt.subplots(1,3,figsize=(16,4))
    for k in range(n): axes[0].plot(times,T_arr[k],"-o",color=cmap(k/n),lw=1.6,ms=3,label=f"I{k}")
    axes[0].set_xlabel("t (s)"); axes[0].set_ylabel("Local gap (°C)"); axes[0].set_title("Gap over time"); axes[0].legend(fontsize=7); axes[0].grid(alpha=.3)
    for k in range(n): axes[1].plot(times,A_arr[k],"-o",color=cmap(k/n),lw=1.6,ms=3,label=f"I{k}")
    axes[1].set_xlabel("t (s)"); axes[1].set_ylabel("Samples"); axes[1].set_title("Allocation"); axes[1].legend(fontsize=7); axes[1].grid(alpha=.3)
    all_g,all_a=T_arr.ravel(),A_arr.ravel()
    for k in range(n): axes[2].scatter(T_arr[k],A_arr[k],color=cmap(k/n),s=30,alpha=.8,edgecolors="k",lw=.3,label=f"I{k}")
    if all_g.max()>1e-6:
        sl=np.polyfit(all_g,all_a,1); gx=np.linspace(0,all_g.max(),80); axes[2].plot(gx,np.polyval(sl,gx),"k--",lw=1.1)
    axes[2].set_xlabel("Gap (°C)"); axes[2].set_ylabel("Samples"); axes[2].set_title("Gap→Allocation"); axes[2].legend(fontsize=7); axes[2].grid(alpha=.3)
    fig.suptitle(title,fontsize=11); plt.tight_layout(); plt.show()


## E8. Config & Runners

EXP_CFG=dict(N=100,dt=1e-3,t_end=100.,T_left=90.,T_right=90.,T_init=20.)
NT_SAVE=21
PROFILE_TIMES=[1.,10.,50.,100.]

MLP_KW=dict(epochs=3000,patience=300,hidden=128,depth=4)
GPR_MAX=5000

ADAP_RI=5.0
ADAP_N_SI=5
ADAP_BUDGET=150
ADAP_MIN_S=5

SOURCE_CONFIGS={
#     2:dict(locs=[0.030,0.070],               temps=[85.,90.]),
#     3:dict(locs=[0.025,0.050,0.075],         temps=[80.,95.,75.]),
#     4:dict(locs=[0.020,0.040,0.060,0.080],   temps=[85.,90.,80.,95.]),
    
    2: dict(locs=[0.00, 0.1],                   temps=[90., 90.]),
    3: dict(locs=[0.00, 0.05, 0.1],                   temps=[90., 90., 90.]),
    4: dict(locs=[0.00, 0.03, 0.070, 0.01],                   temps=[90., 90., 90., 90.]),
}
print("Config ready.")


def run_uniform_exp(n_sources):
    sc=SOURCE_CONFIGS[n_sources]; src_x=sc["locs"]; src_t=sc["temps"]
    tag=f"{n_sources}-source uniform"; seed_everything(42)
    print(f"\n{'='*60}\n  {tag.upper()}\n{'='*60}")
    cfg=RodConfigSrc(**EXP_CFG,source_locs=tuple(src_x),source_temps=tuple(src_t))
    x,t,Thist,_=simulate_heat_ftcs_src(cfg,np.linspace(0.,cfg.t_end,NT_SAVE))
    plot_field_src(x,t,Thist,f"Ground-truth — {tag}",src_x=src_x)
    splits=split_random_frac(x,t,Thist,seed=42); data=splits["data"]
    (Xtr,ytr)=data["train"]; (Xvs,yvs)=data["val_seen"]; (Xvu,yvu)=data["val_unseen"]; (Xte,yte)=data["test_unseen"]
    scaler=StandardScaler().fit(Xtr); Xs={k:scaler.transform(v[0]) for k,v in data.items()}
    mlp,hist=train_mlp(Xs["train"],ytr,Xs["val_seen"],yvs,Xs["val_unseen"],yvu,**MLP_KW)
    mlp.eval(); mlp_fn=mlp_predict_fn(mlp,scaler); plot_train_curves(hist,f"MLP — {tag}")
    gpr=train_gpr(Xs["train"],ytr,GPR_MAX); gpr_fn=gpr_predict_fn(gpr,scaler)
    metrics={}
    for sn,X,y in [("val_seen",Xvs,yvs),("val_unseen",Xvu,yvu),("test_unseen",Xte,yte)]:
        yh_m,_=mlp_fn(X); yh_g,_=gpr_fn(X); metrics[f"MLP/{sn}"]=regression_metrics(y,yh_m); metrics[f"GPR/{sn}"]=regression_metrics(y,yh_g)
        print(f"  MLP {sn:12s} RMSE={metrics[f'MLP/{sn}']['RMSE']:.4f}"); print(f"  GPR {sn:12s} RMSE={metrics[f'GPR/{sn}']['RMSE']:.4f}")
    Xx,Xt=np.meshgrid(x,t); Xfull=np.stack([Xx.ravel(),Xt.ravel()],1)
    mf,_=mlp_fn(Xfull); gf,gs=gpr_fn(Xfull)
    mg=mf.reshape(len(t),len(x)); gg=gf.reshape(len(t),len(x)); sg=gs.reshape(len(t),len(x))
    fig,axes=plt.subplots(2,3,figsize=(18,8))
    for ax,(Z,ttl) in zip(axes.flat,[(mg,"MLP pred"),(np.abs(mg-Thist),"MLP |error|"),(gg,"GPR pred"),(np.abs(gg-Thist),"GPR |error|"),(sg,"GPR std"),(Thist,"Truth")]):
        plot_field_src(x,t,Z,ttl,src_x=src_x,ax=ax)
    fig.suptitle(tag,fontsize=12); plt.tight_layout(); plt.show()
    plot_profiles_src(x,t,Thist,mlp_fn,gpr_fn,PROFILE_TIMES,src_x,f"Profiles — {tag}")
    return dict(x=x,t=t,Thist=Thist,splits=splits,scaler=scaler,mlp=mlp,gpr=gpr,mlp_fn=mlp_fn,gpr_fn=gpr_fn,metrics=metrics,n_train=len(ytr))


def run_adaptive_exp(n_sources,ref_splits,ref_x,ref_t,ref_Thist):
    sc=SOURCE_CONFIGS[n_sources]; src_x=sc["locs"]; src_t=sc["temps"]
    tag=f"{n_sources}-source adaptive"; seed_everything(42)
    print(f"\n{'='*60}\n  {tag.upper()}\n{'='*60}")
    cfg=RodConfigSrc(**EXP_CFG,source_locs=tuple(src_x),source_temps=tuple(src_t))
    sim=RodSimulationSrc(cfg); sampler=AdaptiveSampler(cfg.t_end,ADAP_RI,ADAP_N_SI); mc_env=McCormickEnvelope()
    for t_rec in sampler.record_times:
        T=sim.advance_to(t_rec); sampler.record(mc_env.compute_full(sim.x,T,t_rec))
        if t_rec%20<1e-9 or t_rec==0: print(f"  t={t_rec:6.1f}s  max_gap={sampler.envelope_history[-1].max_gap:.3f}°C")
    sampler.allocate_global_budget(ADAP_BUDGET+50,ADAP_MIN_S)
    gap_vs_allocation_mpl(sampler,f"Gap vs Allocation — {tag}")
    snap_times=[0,10,25,50,75,100]; fig,axes=plt.subplots(2,3,figsize=(18,9))
    for ax,tv in zip(axes.flat,snap_times): plot_subdomain_envelopes_src(sampler,time_value=tv,ax=ax)
    fig.suptitle(f"Sub-domain envelopes — {tag}",fontsize=12); plt.tight_layout(); plt.show()
    xs,ts,Ts=[],[],[]
    for plan in sampler.sampling_plans:
        env=next(e for e in sampler.envelope_history if abs(e.time-plan.time)<1e-6); seen=set()
        for k,sx_arr in plan.sample_indices.items():
            for xv in sx_arr:
                key=round(xv,10)
                if key not in seen:
                    seen.add(key); xs.append(float(xv)); ts.append(float(plan.time)); Ts.append(float(np.interp(xv,env.x,env.T)))
    Xtr_a=np.stack([xs,ts],1); ytr_a=np.array(Ts); print(f"  Adaptive pts: {len(ytr_a):,}")
    (Xvs,yvs)=ref_splits["data"]["val_seen"]; (Xvu,yvu)=ref_splits["data"]["val_unseen"]; (Xte,yte)=ref_splits["data"]["test_unseen"]
    scaler_a=StandardScaler().fit(Xtr_a); Xtr_s=scaler_a.transform(Xtr_a); Xvs_s=scaler_a.transform(Xvs); Xvu_s=scaler_a.transform(Xvu)
    mlp_a,hist_a=train_mlp(Xtr_s,ytr_a,Xvs_s,yvs,Xvu_s,yvu,**MLP_KW)
    mlp_a.eval(); mlp_fn_a=mlp_predict_fn(mlp_a,scaler_a); plot_train_curves(hist_a,f"MLP — {tag}")
    gpr_a=train_gpr(Xtr_s,ytr_a,GPR_MAX); gpr_fn_a=gpr_predict_fn(gpr_a,scaler_a)
    metrics={}
    for sn,X,y in [("val_seen",Xvs,yvs),("val_unseen",Xvu,yvu),("test_unseen",Xte,yte)]:
        yh_m,_=mlp_fn_a(X); yh_g,_=gpr_fn_a(X); metrics[f"MLP/{sn}"]=regression_metrics(y,yh_m); metrics[f"GPR/{sn}"]=regression_metrics(y,yh_g)
        print(f"  MLP {sn:12s} RMSE={metrics[f'MLP/{sn}']['RMSE']:.4f}"); print(f"  GPR {sn:12s} RMSE={metrics[f'GPR/{sn}']['RMSE']:.4f}")
    Xx,Xt=np.meshgrid(ref_x,ref_t); Xfull=np.stack([Xx.ravel(),Xt.ravel()],1)
    mf,_=mlp_fn_a(Xfull); gf,gs=gpr_fn_a(Xfull)
    mg=mf.reshape(len(ref_t),len(ref_x)); gg=gf.reshape(len(ref_t),len(ref_x)); sg=gs.reshape(len(ref_t),len(ref_x))
    fig,axes=plt.subplots(2,3,figsize=(18,8))
    for ax,(Z,ttl) in zip(axes.flat,[(mg,"MLP pred"),(np.abs(mg-ref_Thist),"MLP |error|"),(gg,"GPR pred"),(np.abs(gg-ref_Thist),"GPR |error|"),(sg,"GPR std"),(ref_Thist,"Truth")]):
        plot_field_src(ref_x,ref_t,Z,ttl,src_x=src_x,ax=ax)
    fig.suptitle(tag,fontsize=12); plt.tight_layout(); plt.show()
    plot_profiles_src(ref_x,ref_t,ref_Thist,mlp_fn_a,gpr_fn_a,PROFILE_TIMES,src_x,f"Profiles — {tag}")
    return dict(scaler=scaler_a,mlp=mlp_a,gpr=gpr_a,mlp_fn=mlp_fn_a,gpr_fn=gpr_fn_a,metrics=metrics,sampler=sampler,n_train=len(ytr_a),X_train=Xtr_a,y_train=ytr_a)


## B1. BisectionSamplerHeat

class BisectionSamplerHeat:
    def __init__(self,x,t,Thist,budget=400,gap_measure="mccormick_x"):
        assert Thist.shape==(len(t),len(x))
        self.x=x; self.t=t; self.Thist=Thist; self.budget=budget; self.gap_measure=gap_measure
        self._interp=RegularGridInterpolator((t,x),Thist,method="linear",bounds_error=False,fill_value=None)
        self._mc=McCormickEnvelope(); self.domains=[]; self.gaps=[]; self.history_n=[]; self.history_max_gap=[]
    def compute_gap(self,x1,x2,t1,t2):
        eps=1e-12; xi_m=(self.x>=x1-eps)&(self.x<=x2+eps); ti_m=(self.t>=t1-eps)&(self.t<=t2+eps)
        sub=self.Thist[np.ix_(ti_m,xi_m)]
        if sub.size==0: return eps
        if self.gap_measure=="range": return float(sub.max()-sub.min())
        xx=self.x[xi_m]
        if len(xx)<3: return float(sub.max()-sub.min())
        if self.gap_measure=="mccormick_x":
            t_mid=(t1+t2)/2.; ti_inds=np.where(ti_m)[0]; ti_mid=ti_inds[int(np.argmin(np.abs(self.t[ti_m]-t_mid)))]
            return float(self._mc.compute(xx,self.Thist[ti_mid,xi_m])["max_gap"])
        if self.gap_measure=="mccormick_max":
            ti_inds=np.where(ti_m)[0]; step=max(1,len(ti_inds)//20)
            return float(max(self._mc.compute(xx,self.Thist[ti_idx,xi_m])["max_gap"] for ti_idx in ti_inds[::step]))
        raise ValueError(f"Unknown gap_measure: {self.gap_measure!r}")
    def run(self):
        x_lo,x_hi=float(self.x[0]),float(self.x[-1]); t_lo,t_hi=float(self.t[0]),float(self.t[-1])
        domains=[(x_lo,x_hi,t_lo,t_hi)]; gaps=[self.compute_gap(x_lo,x_hi,t_lo,t_hi)]
        self.history_n=[1]; self.history_max_gap=[gaps[0]]; target=self.budget-3
        while len(domains)<target:
            idx=int(np.argmax(gaps)); x1,x2,t1,t2=domains.pop(idx); gaps.pop(idx)
            mx=(x1+x2)/2.; mt=(t1+t2)/2.
            for ch in [(x1,mx,t1,mt),(mx,x2,t1,mt),(x1,mx,mt,t2),(mx,x2,mt,t2)]: domains.append(ch); gaps.append(self.compute_gap(*ch))
            self.history_n.append(len(domains)); self.history_max_gap.append(float(max(gaps)))
        corners=[]
        for (x1,x2,t1,t2) in domains: corners.extend([[x1,t1],[x1,t2],[x2,t1],[x2,t2]])
        sxt=np.unique(np.array(corners),axis=0); sxt=sxt[np.lexsort((sxt[:,1],sxt[:,0]))]
        sT=self._interp(sxt[:,[1,0]]); self.domains=domains; self.gaps=gaps
        print(f"  Bisection: {len(domains)} rects → {len(sxt)} unique corners  [gap={self.gap_measure!r}]")
        return sxt, sT, domains, gaps


## B2. Bisection Visualisation Helpers

def plot_bisection_partition(bsamp,src_x=None,title="Bisection Partition"):
    fig,axes=plt.subplots(1,2,figsize=(16,5)); ax=axes[0]
    patches=[Rectangle((x1,t1),x2-x1,t2-t1) for (x1,x2,t1,t2) in bsamp.domains]
    coll=PatchCollection(patches,cmap="YlOrRd",alpha=0.72,edgecolor="k",linewidth=0.25)
    coll.set_array(np.array(bsamp.gaps)); ax.add_collection(coll); plt.colorbar(coll,ax=ax,label="Gap (°C)")
    if src_x:
        for i,sx in enumerate(src_x): ax.axvline(sx,color="deepskyblue",lw=1.5,ls="--",alpha=.85,label="source" if i==0 else None)
        ax.legend(fontsize=8)
    ax.set_xlim(bsamp.x[0],bsamp.x[-1]); ax.set_ylim(bsamp.t[0],bsamp.t[-1]); ax.set_xlabel("x (m)"); ax.set_ylabel("t (s)")
    ax.set_title(f"{len(bsamp.domains)} rects"); axes[1].plot(bsamp.history_n,bsamp.history_max_gap,"-o",ms=3,color="crimson",lw=1.5)
    axes[1].set_xlabel("N rectangles"); axes[1].set_ylabel("Max gap (°C)"); axes[1].set_title("Gap decay"); axes[1].grid(alpha=.3)
    fig.suptitle(title,fontsize=12); plt.tight_layout(); plt.show()

def plot_bisection_samples(sxt,sT,bsamp,src_x=None,title="Bisection Samples"):
    fig,axes=plt.subplots(1,2,figsize=(15,4))
    sc=axes[0].scatter(sxt[:,0],sxt[:,1],c=sT,s=14,alpha=.75,cmap="plasma"); plt.colorbar(sc,ax=axes[0],label="T (°C)")
    if src_x:
        for sx in src_x: axes[0].axvline(sx,color="lime",lw=1.5,ls="--",alpha=.85)
    axes[0].set_xlabel("x (m)"); axes[0].set_ylabel("t (s)"); axes[0].set_title(f"{len(sxt)} corners")
    axes[1].hist2d(sxt[:,0],sxt[:,1],bins=30,cmap="Blues"); plt.colorbar(axes[1].collections[0],ax=axes[1],label="Count")
    if src_x:
        for sx in src_x: axes[1].axvline(sx,color="red",lw=1.5,ls="--")
    axes[1].set_xlabel("x (m)"); axes[1].set_ylabel("t (s)"); axes[1].set_title("2-D density")
    fig.suptitle(title,fontsize=11); plt.tight_layout(); plt.show()

def plot_bisection_3d(sxt,sT,bsamp,src_x=None,title=None):
    if title is None: title=f"Bisection 3-D — {len(sxt)} corners"
    T_floor=float(sT.min())-2.; T_max=float(sT.max())
    gap_arr=bsamp.gaps; g_min,g_max=min(gap_arr),max(gap_arr)
    cscale=pc.get_colorscale("YlOrRd")
    def gap_color(g):
        frac=(g-g_min)/(g_max-g_min) if g_max>g_min else 0.5; return pc.sample_colorscale(cscale,frac)[0]
    traces=[]
    traces.append(go.Scatter3d(x=sxt[:,0],y=sxt[:,1],z=sT,mode="markers",
        marker=dict(size=4,color=sT,colorscale="Plasma",colorbar=dict(title="T (°C)",thickness=14,x=1.02),opacity=0.88,line=dict(color="black",width=0.3)),
        name="Sample corners",hovertemplate="x=%{x:.4f}m<br>t=%{y:.1f}s<br>T=%{z:.2f}°C<extra></extra>"))
    rx,rt,rz=[],[],[]
    for (x1,x2,t1,t2) in bsamp.domains:
        for xv,tv in [(x1,t1),(x2,t1),(x2,t2),(x1,t2),(x1,t1)]: rx.append(xv); rt.append(tv); rz.append(T_floor)
        rx.append(None); rt.append(None); rz.append(None)
    traces.append(go.Scatter3d(x=rx,y=rt,z=rz,mode="lines",line=dict(color="rgba(90,90,90,0.35)",width=1),name="Leaf rectangles",hoverinfo="skip"))
    vx,vt,vz=[],[],[]
    for (xi,ti),Ti in zip(sxt,sT): vx+=[xi,xi,None]; vt+=[ti,ti,None]; vz+=[T_floor,float(Ti),None]
    traces.append(go.Scatter3d(x=vx,y=vt,z=vz,mode="lines",line=dict(color="rgba(150,150,150,0.20)",width=1),name="Drop lines",hoverinfo="skip"))
    if src_x:
        t_lo,t_hi=float(bsamp.t[0]),float(bsamp.t[-1])
        for sx in src_x: traces.append(go.Mesh3d(x=[sx,sx,sx,sx],y=[t_lo,t_lo,t_hi,t_hi],z=[T_floor,T_max+2,T_max+2,T_floor],opacity=0.12,color="red",name=f"src x={sx:.3f}m",hoverinfo="skip"))
    for (x1,x2,t1,t2),g in zip(bsamp.domains,gap_arr):
        col=gap_color(g); traces.append(go.Mesh3d(x=[x1,x2,x2,x1],y=[t1,t1,t2,t2],z=[T_floor]*4,i=[0,0],j=[1,2],k=[2,3],opacity=0.55,color=col,showscale=False,hoverinfo="skip"))
    fig=go.Figure(data=traces)
    fig.update_layout(title=dict(text=title,font=dict(size=14)),
        scene=dict(xaxis=dict(title="x (m)"),yaxis=dict(title="t (s)"),zaxis=dict(title="T (°C)"),
                   camera=dict(eye=dict(x=1.6,y=-1.6,z=1.1)),aspectmode="manual",aspectratio=dict(x=1.5,y=2.0,z=1.2)),
        legend=dict(x=0.01,y=0.99,font=dict(size=11)),width=1050,height=700,margin=dict(l=0,r=0,b=0,t=50))
    return fig

def plot_sampling_comparison(sxt_bsect,sxt_adapt,Xtr_unif,src_x=None,title="Sampling comparison"):
    fig,axes=plt.subplots(1,3,figsize=(18,4))
    for ax,pts,ttl in [(axes[0],sxt_bsect,f"Bisection ({len(sxt_bsect)} pts)"),
                        (axes[1],sxt_adapt, f"Adaptive ({len(sxt_adapt)} pts)"),
                        (axes[2],Xtr_unif,  f"Uniform ({len(Xtr_unif)} pts)")]:
        ax.hist2d(pts[:,0],pts[:,1],bins=25,cmap="Blues"); plt.colorbar(ax.collections[0],ax=ax,label="Count")
        if src_x:
            for sx in src_x: ax.axvline(sx,color="red",lw=1.5,ls="--",alpha=.7)
        ax.set_xlabel("x (m)"); ax.set_ylabel("t (s)"); ax.set_title(ttl)
    plt.suptitle(title,fontsize=11); plt.tight_layout(); plt.show()


## B3. run_bisection_exp

BISECT_BUDGET=150; BISECT_GAP="mccormick_x"

def run_bisection_exp(n_sources,ref_splits,ref_x,ref_t,ref_Thist,budget=BISECT_BUDGET,gap_measure=BISECT_GAP):
    sc=SOURCE_CONFIGS[n_sources]; src_x_locs=sc["locs"]; tag=f"{n_sources}-source bisection"; seed_everything(42)
    print(f"\n{'='*60}\n  {tag.upper()}\n{'='*60}")
    bsamp=BisectionSamplerHeat(ref_x,ref_t,ref_Thist,budget=budget,gap_measure=gap_measure)
    sxt,sT,domains,gaps=bsamp.run()
    plot_bisection_partition(bsamp,src_x=src_x_locs,title=f"Bisection partition — {tag}")
    plot_bisection_samples(sxt,sT,bsamp,src_x=src_x_locs,title=f"Bisection samples — {tag}")
    plot_bisection_3d(sxt,sT,bsamp,src_x=src_x_locs,title=f"Bisection 3-D — {tag}").show()
    (Xvs,yvs)=ref_splits["data"]["val_seen"]; (Xvu,yvu)=ref_splits["data"]["val_unseen"]; (Xte,yte)=ref_splits["data"]["test_unseen"]
    scaler_b=StandardScaler().fit(sxt); Xtr_s=scaler_b.transform(sxt); Xvs_s=scaler_b.transform(Xvs); Xvu_s=scaler_b.transform(Xvu)
    mlp_b,hist_b=train_mlp(Xtr_s,sT,Xvs_s,yvs,Xvu_s,yvu,**MLP_KW)
    mlp_b.eval(); mlp_fn_b=mlp_predict_fn(mlp_b,scaler_b); plot_train_curves(hist_b,f"MLP — {tag}")
    gpr_b=train_gpr(Xtr_s,sT,GPR_MAX); gpr_fn_b=gpr_predict_fn(gpr_b,scaler_b)
    metrics={}
    for sn,X,y in [("val_seen",Xvs,yvs),("val_unseen",Xvu,yvu),("test_unseen",Xte,yte)]:
        yh_m,_=mlp_fn_b(X); yh_g,_=gpr_fn_b(X); metrics[f"MLP/{sn}"]=regression_metrics(y,yh_m); metrics[f"GPR/{sn}"]=regression_metrics(y,yh_g)
        print(f"  MLP {sn:12s}  RMSE={metrics[f'MLP/{sn}']['RMSE']:.4f}"); print(f"  GPR {sn:12s}  RMSE={metrics[f'GPR/{sn}']['RMSE']:.4f}")
    Xx,Xt=np.meshgrid(ref_x,ref_t); Xfull=np.stack([Xx.ravel(),Xt.ravel()],1)
    mf,_=mlp_fn_b(Xfull); gf,gs=gpr_fn_b(Xfull)
    mg=mf.reshape(len(ref_t),len(ref_x)); gg=gf.reshape(len(ref_t),len(ref_x)); sg=gs.reshape(len(ref_t),len(ref_x))
    fig,axes=plt.subplots(2,3,figsize=(18,8))
    for ax,(Z,ttl) in zip(axes.flat,[(mg,"MLP pred"),(np.abs(mg-ref_Thist),"MLP |error|"),(gg,"GPR pred"),(np.abs(gg-ref_Thist),"GPR |error|"),(sg,"GPR std"),(ref_Thist,"Truth")]):
        plot_field_src(ref_x,ref_t,Z,ttl,src_x=src_x_locs,ax=ax)
    fig.suptitle(tag,fontsize=12); plt.tight_layout(); plt.show()
    plot_profiles_src(ref_x,ref_t,ref_Thist,mlp_fn_b,gpr_fn_b,PROFILE_TIMES,src_x_locs,f"Profiles — {tag}")
    return dict(scaler=scaler_b,mlp=mlp_b,gpr=gpr_b,mlp_fn=mlp_fn_b,gpr_fn=gpr_fn_b,metrics=metrics,sampler=bsamp,n_train=len(sT),X_train=sxt,y_train=sT)



## C1. Acquisition Functions

def acq_std(mu,std,gap_weight,T_best,xi=0.0):     return std.copy()
def acq_std_gap(mu,std,gap_weight,T_best,xi=0.0): return std*gap_weight
def acq_ei(mu,std,gap_weight,T_best,xi=0.0):
    sigma=np.maximum(std,1e-12); Z=(mu-T_best-xi)/sigma
    return np.maximum((mu-T_best-xi)*scipy_norm.cdf(Z)+sigma*scipy_norm.pdf(Z),0.)
def acq_ei_gap(mu,std,gap_weight,T_best,xi=0.0):
    sigma_inf=np.maximum(std*gap_weight,1e-12); Z=(mu-T_best-xi)/sigma_inf
    return np.maximum((mu-T_best-xi)*scipy_norm.cdf(Z)+sigma_inf*scipy_norm.pdf(Z),0.)
ACQUISITION_STRATEGIES:Dict[str,callable]={"STD":acq_std,"STD×gap":acq_std_gap,"EI":acq_ei,"EI×gap":acq_ei_gap}
STRATEGY_COLORS={"STD":"C0","STD×gap":"C2","EI":"C3","EI×gap":"C4"}
print("Acquisition functions:", list(ACQUISITION_STRATEGIES.keys()))


## C2. Gap-weight Grid Builder

def build_gap_weight_grid(X_grid_2d,sampler):
    plans=sampler.sampling_plans; plan_times=np.array([p.time for p in plans])
    x_arr=sampler.envelope_history[0].x; x_lo,x_hi=float(x_arr[0]),float(x_arr[-1])
    n_si=sampler.n_intervals; edges=np.linspace(x_lo,x_hi,n_si+1)
    raw_gap=np.zeros(len(X_grid_2d),dtype=float)
    for i,(xi,ti) in enumerate(X_grid_2d):
        s_idx=int(np.argmin(np.abs(plan_times-ti))); plan=plans[s_idx]
        k=int(np.clip(int(np.searchsorted(edges,xi,side="right"))-1,0,n_si-1))
        raw_gap[i]=plan.local_envelopes[k]["max_gap"]
    total=raw_gap.sum()
    return (raw_gap/total)*len(raw_gap) if total>1e-10 else np.ones(len(raw_gap))


## C3. Visualisation Helpers

# ── gap_weight heatmap ────────────────────────────────────────────────────────
def plot_gap_weight_map(X_grid_2d,gap_weight,grid_nx,grid_nt,src_x,title):
    gx=np.unique(X_grid_2d[:,0]); gt=np.unique(X_grid_2d[:,1])
    gw=gap_weight.reshape(grid_nt,grid_nx)
    fig,ax=plt.subplots(figsize=(9,3.8))
    im=ax.imshow(gw,aspect="auto",origin="lower",extent=[gx[0],gx[-1],gt[0],gt[-1]],cmap="YlOrRd")
    plt.colorbar(im,ax=ax,label="gap_weight (norm.)")
    for sx in (src_x or []): ax.axvline(sx,color="blue",lw=1.5,ls="--",alpha=.8)
    ax.set_xlabel("x (m)"); ax.set_ylabel("t (s)"); ax.set_title(title)
    plt.tight_layout(); plt.show()

# ── per-step acquisition map ──────────────────────────────────────────────────
def plot_acquisition_step(X_grid_2d,acq_vals,std_vals,X_train,x_next,grid_nx,grid_nt,ref_x,ref_t,ref_Thist,step,strategy,src_x=None):
    gx=np.unique(X_grid_2d[:,0]); gt=np.unique(X_grid_2d[:,1]); ext=[gx[0],gx[-1],gt[0],gt[-1]]
    fig,axes=plt.subplots(1,3,figsize=(18,4))
    im0=axes[0].imshow(acq_vals.reshape(grid_nt,grid_nx),aspect="auto",origin="lower",extent=ext,cmap="hot")
    plt.colorbar(im0,ax=axes[0],label="Acquisition"); axes[0].set_title(f"{strategy} — step {step}"); axes[0].set_xlabel("x (m)"); axes[0].set_ylabel("t (s)")
    for sx in (src_x or []): axes[0].axvline(sx,color="cyan",lw=1.2,ls="--",alpha=.8)
    if x_next is not None: axes[0].scatter(x_next[0,0],x_next[0,1],marker="*",s=220,color="lime",zorder=10,label="next"); axes[0].legend(fontsize=8)
    im1=axes[1].imshow(std_vals.reshape(grid_nt,grid_nx),aspect="auto",origin="lower",extent=ext,cmap="viridis")
    plt.colorbar(im1,ax=axes[1],label="GPR σ (°C)"); axes[1].set_title(f"GPR σ — step {step}"); axes[1].set_xlabel("x (m)"); axes[1].set_ylabel("t (s)")
    axes[1].scatter(X_train[:,0],X_train[:,1],s=5,alpha=.4,color="white",zorder=5)
    if x_next is not None: axes[1].scatter(x_next[0,0],x_next[0,1],marker="*",s=220,color="red",zorder=10)
    for sx in (src_x or []): axes[1].axvline(sx,color="orange",lw=1.2,ls="--",alpha=.7)
    plot_field_src(ref_x,ref_t,ref_Thist,"Ground truth",src_x=src_x,ax=axes[2])
    plt.tight_layout(); plt.show()

# ── 4-strategy acquisition snapshot ──────────────────────────────────────────
def plot_all_acquisitions_snapshot(mu,std_map,gap_weight,T_best,X_grid_2d,grid_nx,grid_nt,x_next_dict,src_x,step,title="Acquisition snapshot"):
    fig,axes=plt.subplots(2,2,figsize=(16,9))
    for ax,(name,acq_fn) in zip(axes.flat,ACQUISITION_STRATEGIES.items()):
        acq=acq_fn(mu,std_map,gap_weight,T_best)
        im=ax.imshow(acq.reshape(grid_nt,grid_nx),aspect="auto",origin="lower",
                     extent=[X_grid_2d[:,0].min(),X_grid_2d[:,0].max(),X_grid_2d[:,1].min(),X_grid_2d[:,1].max()],cmap="hot")
        plt.colorbar(im,ax=ax,label=name); ax.set_title(f"{name}  (step {step})"); ax.set_xlabel("x (m)"); ax.set_ylabel("t (s)")
        for sx in (src_x or []): ax.axvline(sx,color="cyan",lw=1.2,ls="--",alpha=.8)
        xn=x_next_dict.get(name)
        if xn is not None: ax.scatter(xn[0,0],xn[0,1],marker="*",s=260,color=STRATEGY_COLORS.get(name,"white"),zorder=10,label="next"); ax.legend(fontsize=8)
    fig.suptitle(title,fontsize=13); plt.tight_layout(); plt.show()

# ── sample trajectory ─────────────────────────────────────────────────────────
def plot_sample_trajectory(uncert_results,n_warm,ref_x,ref_t,src_x,title):
    strategies=list(uncert_results.keys()); n=len(strategies)
    fig,axes=plt.subplots(1,n,figsize=(5*n,4),sharey=True)
    if n==1: axes=[axes]
    for ax,strat in zip(axes,strategies):
        res=uncert_results[strat]; X_all=res["X_train"]; X_warm=X_all[:n_warm]; X_ref=X_all[n_warm:]
        ax.scatter(X_warm[:,0],X_warm[:,1],s=10,c="grey",alpha=.4,zorder=3,label=f"warm ({n_warm})")
        if len(X_ref)>0:
            sc=ax.scatter(X_ref[:,0],X_ref[:,1],s=25,c=np.arange(len(X_ref)),cmap="plasma",alpha=.85,zorder=5,edgecolors="k",lw=.3,label=f"refined ({len(X_ref)})")
            plt.colorbar(sc,ax=ax,label="Step")
        for sx in (src_x or []): ax.axvline(sx,color="red",lw=1.3,ls="--",alpha=.7)
        ax.set_xlim(ref_x[0],ref_x[-1]); ax.set_ylim(ref_t[0],ref_t[-1])
        ax.set_xlabel("x (m)"); ax.set_title(strat); ax.legend(fontsize=7)
        if ax is axes[0]: ax.set_ylabel("t (s)")
    fig.suptitle(title,fontsize=12); plt.tight_layout(); plt.show()

# ── full density comparison ───────────────────────────────────────────────────
def plot_full_density_comparison(uncert_results,bisect_X,adapt_X,unif_X,src_x,title):
    panels=([(s,uncert_results[s]["X_train"]) for s in uncert_results]
            +[("bisection",bisect_X),("adaptive",adapt_X),("uniform",unif_X)])
    n=len(panels); ncols=4; nrows=math.ceil(n/ncols)
    fig,axes=plt.subplots(nrows,ncols,figsize=(5.5*ncols,4.5*nrows))
    axes_flat=axes.flat if hasattr(axes,"flat") else [axes]
    for ax,(name,pts) in zip(axes_flat,panels):
        h=ax.hist2d(pts[:,0],pts[:,1],bins=25,cmap="Blues"); plt.colorbar(h[3],ax=ax,label="Count")
        for sx in (src_x or []): ax.axvline(sx,color="red",lw=1.3,ls="--",alpha=.7)
        ax.set_xlabel("x (m)"); ax.set_ylabel("t (s)"); ax.set_title(f"{name} ({len(pts)} pts)")
    for ax in list(axes_flat)[n:]: ax.set_visible(False)
    fig.suptitle(title,fontsize=13); plt.tight_layout(); plt.show()

# ── RMSE convergence ──────────────────────────────────────────────────────────
def plot_rmse_convergence(history,title):
    fig,ax=plt.subplots(figsize=(9,4))
    for strat,vals in history.items():
        steps=[v["step"] for v in vals]; rmse=[v["test_rmse"] for v in vals]
        ax.plot(steps,rmse,"-o",ms=5,lw=2,color=STRATEGY_COLORS.get(strat,"k"),label=strat)
    ax.set_xlabel("Refinement step"); ax.set_ylabel("Test RMSE (°C)"); ax.set_title(title)
    ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

# ── per-strategy error fields ─────────────────────────────────────────────────
def plot_strategy_error_fields(uncert_results,ref_x,ref_t,ref_Thist,src_x,title):
    strategies=list(uncert_results.keys()); n=len(strategies)
    fig,axes=plt.subplots(n,3,figsize=(18,4.5*n))
    if n==1: axes=axes[np.newaxis,:]
    Xx,Xt=np.meshgrid(ref_x,ref_t); Xfull=np.stack([Xx.ravel(),Xt.ravel()],1)
    for row,strat in enumerate(strategies):
        gpr_fn=uncert_results[strat]["gpr_fn"]; gf,gs=gpr_fn(Xfull)
        gg=gf.reshape(len(ref_t),len(ref_x)); sg=gs.reshape(len(ref_t),len(ref_x)); err=np.abs(gg-ref_Thist)
        for col,(Z,lbl) in enumerate([(gg,"GPR pred (°C)"),(err,"GPR |error| (°C)"),(sg,"GPR σ (°C)")]):
            plot_field_src(ref_x,ref_t,Z,f"{strat} — {lbl}",cbar_label=lbl,src_x=src_x,ax=axes[row,col])
        axes[row,0].set_ylabel(strat,fontsize=11,fontweight="bold")
    fig.suptitle(title,fontsize=13); plt.tight_layout(); plt.show()

# ── profile comparison ────────────────────────────────────────────────────────
def plot_strategy_profiles(uncert_results,ref_x,ref_t,ref_Thist,times_plot,src_x,title):
    n_times=len(times_plot); fig,axes=plt.subplots(1,n_times,figsize=(4.5*n_times,4),sharey=True)
    if n_times==1: axes=[axes]
    cmap=plt.cm.get_cmap("tab10",len(uncert_results))
    for col,pt in enumerate(times_plot):
        ki=int(np.argmin(np.abs(ref_t-pt))); Xq=np.stack([ref_x,np.full_like(ref_x,ref_t[ki])],1); ax=axes[col]
        ax.plot(ref_x,ref_Thist[ki],"k-",lw=2.5,label="Truth",zorder=10)
        for si,(strat,res) in enumerate(uncert_results.items()):
            m,s=res["gpr_fn"](Xq); ax.plot(ref_x,m,"--",lw=1.6,color=cmap(si),label=strat,zorder=5)
            ax.fill_between(ref_x,m-2*s,m+2*s,alpha=.08,color=cmap(si))
        for sx in (src_x or []): ax.axvline(sx,color="grey",lw=1,ls=":",alpha=.7)
        ax.set_title(f"t = {ref_t[ki]:.1f} s"); ax.set_xlabel("x (m)")
        if col==0: ax.set_ylabel("T (°C)")
        ax.legend(fontsize=7)
    fig.suptitle(title,fontsize=12); plt.tight_layout(); plt.show()

# ── pred-vs-true scatter ──────────────────────────────────────────────────────
def plot_strategy_pred_vs_true(uncert_results,Xte,yte,title):
    n=len(uncert_results); fig,axes=plt.subplots(1,n,figsize=(4.5*n,4.5))
    if n==1: axes=[axes]
    for ax,(strat,res) in zip(axes,uncert_results.items()):
        yp,_=res["gpr_fn"](Xte); rmse=math.sqrt(mean_squared_error(yte,yp)); r2=r2_score(yte,yp)
        ax.scatter(yte,yp,s=6,alpha=.35,color=STRATEGY_COLORS.get(strat,"C0"))
        mn,mx=min(yte.min(),yp.min()),max(yte.max(),yp.max()); ax.plot([mn,mx],[mn,mx],"r--",lw=1)
        ax.set_xlabel("True T (°C)"); ax.set_ylabel("Pred T (°C)"); ax.set_title(f"{strat}\nRMSE={rmse:.3f}  R²={r2:.3f}")
    fig.suptitle(title,fontsize=12); plt.tight_layout(); plt.show()

# ── convergence all methods ───────────────────────────────────────────────────
def plot_convergence_all_methods(uniform_all,adaptive_all,bisect_all,uncertainty_all,n_sources,split="test_unseen"):
    fig,ax=plt.subplots(figsize=(10,5))
    for _,(res,ls,lbl) in enumerate([("uniform",uniform_all[n_sources],"k--","Uniform"),
                                     ("adaptive",adaptive_all[n_sources],"b--","Adaptive"),
                                     ("bisection",bisect_all[n_sources],"m--","Bisection")]):
        n_tr=res["n_train"]; rmse=res["metrics"].get(f"GPR/{split}",{}).get("RMSE",float("nan"))
        ax.axhline(rmse,ls="--",lw=1.5,color=ls.replace("--","").strip(),label=f"{lbl} ({n_tr} pts)")
        ax.annotate(f"{rmse:.3f}",xy=(n_tr,rmse),fontsize=8)
    for strat,res in uncertainty_all[n_sources].items():
        hist=res["history"]
        if not hist: continue
        n_pts=[h["n_train"] for h in hist]; rmse=[h["test_rmse"] for h in hist]
        ax.plot(n_pts,rmse,"-o",ms=5,lw=2,color=STRATEGY_COLORS.get(strat,"k"),label=f"Uncert[{strat}]")
    ax.set_xlabel("N training points"); ax.set_ylabel("Test RMSE (°C)")
    ax.set_title(f"RMSE convergence — {n_sources} sources [{split}]")
    ax.legend(fontsize=9); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

# ── RMSE heatmap ──────────────────────────────────────────────────────────────
def plot_rmse_heatmap(uniform_all,adaptive_all,bisect_all,uncertainty_all,split="test_unseen",model="GPR"):
    n_src_list=sorted(uniform_all.keys()); uncert_strats=list(ACQUISITION_STRATEGIES.keys())
    col_labels=["uniform","adaptive","bisection"]+[f"uncert\n{s}" for s in uncert_strats]
    matrix=[]
    for n in n_src_list:
        row=[]
        for strat in ["uniform","adaptive","bisection"]:
            rd={"uniform":uniform_all,"adaptive":adaptive_all,"bisection":bisect_all}[strat]
            row.append(rd[n]["metrics"].get(f"{model}/{split}",{}).get("RMSE",float("nan")))
        for s in uncert_strats:
            v=uncertainty_all[n].get(s,{}).get("metrics",{}).get(f"{model}/{split}",{}).get("RMSE",float("nan"))
            row.append(v)
        matrix.append(row)
    mat=np.array(matrix,dtype=float)
    fig,ax=plt.subplots(figsize=(max(10,len(col_labels)*1.6),3.5+0.6*len(n_src_list)))
    sns.heatmap(mat,annot=True,fmt=".3f",cmap="RdYlGn_r",xticklabels=col_labels,yticklabels=[f"{n} src" for n in n_src_list],
                ax=ax,linewidths=.5,linecolor="grey",cbar_kws=dict(label="RMSE (°C)"))
    ax.set_title(f"{model} RMSE heatmap — [{split}]",fontsize=13)
    plt.tight_layout(); plt.show()


### C3b. `plot_uncertainty_3d` (Updated — Reference Style)

def plot_uncertainty_3d(X_train: np.ndarray,
                        T_train: np.ndarray,
                        X_warm:  np.ndarray,
                        strategy: str,
                        sampler:  AdaptiveSampler,
                        src_x:    list = None,
                        title:    str  = None,
                        t_stride: int  = 5) -> go.Figure:
    """
    Interactive 3-D figure for uncertainty-driven active sampling.

    Six trace groups (mirrors the structure of plot_3d_envelopes_with_samples):

      Group 1 — T(x,t)        solid blue lines, one per recorded time slice
      Group 2 — T_under(x,t)  dashed orange lines (McCormick lower envelope)
      Group 3 — T_over(x,t)   dotted green lines  (McCormick upper envelope)
      Group 4 — Warm-start     red circle markers pulled from sampler.sampling_plans
      Group 5 — Refinement     Plasma-coloured diamond markers (colour = step order)
      Group 6 — Source planes  translucent red vertical sheets

    Parameters
    ----------
    X_train  : (N_total, 2)  all training points [x, t]
    T_train  : (N_total,)    corresponding T values
    X_warm   : (N_warm,  2)  warm-start subset (first N_warm rows of X_train)
    strategy : str           strategy name shown in title / legend
    sampler  : AdaptiveSampler  provides envelope_history (T, T_under, T_over)
                                and sampling_plans (warm-start point locations)
    src_x    : list of float    source x-positions (m)
    title    : str, optional
    t_stride : int              render every t_stride-th time slice for Groups 1-3
                                (default 5 keeps trace count manageable for ~50 slices)
    """
    if title is None:
        n_warm   = len(X_warm)
        n_refine = len(X_train) - n_warm
        title    = (f"Uncertainty 3-D — {strategy} | "
                    f"{n_warm} warm-start + {n_refine} refined = {len(X_train)} total")

    colors = {
        "T":       "rgb(31,  119, 180)",   # blue
        "T_under": "rgb(255, 127,  14)",   # orange
        "T_over":  "rgb( 44, 160,  44)",   # green
        "samples": "rgb(214,  39,  40)",   # red  (warm-start markers)
    }

    fig  = go.Figure()
    hist = sampler.envelope_history   # List[EnvelopeData]
    plans= sampler.sampling_plans     # List[SamplingPlan]

    # ── Group 1: T(x,t) curves ───────────────────────────────────────────────
    for i, env in enumerate(hist[::t_stride]):
        fig.add_trace(go.Scatter3d(
            x=env.x, y=[env.time]*len(env.x), z=env.T,
            mode="lines",
            name=f"T(x,t={env.time:.0f}s)",
            line=dict(color=colors["T"], width=3),
            legendgroup="temperature",
            showlegend=(i == 0),
        ))

    # ── Group 2: T_under(x,t) curves ────────────────────────────────────────
    for i, env in enumerate(hist[::t_stride]):
        fig.add_trace(go.Scatter3d(
            x=env.x, y=[env.time]*len(env.x), z=env.T_under,
            mode="lines",
            name="McCormick lower (T_under)",
            line=dict(color=colors["T_under"], width=2, dash="dash"),
            legendgroup="under",
            showlegend=(i == 0),
        ))

    # ── Group 3: T_over(x,t) curves ─────────────────────────────────────────
    for i, env in enumerate(hist[::t_stride]):
        fig.add_trace(go.Scatter3d(
            x=env.x, y=[env.time]*len(env.x), z=env.T_over,
            mode="lines",
            name="McCormick upper (T_over)",
            line=dict(color=colors["T_over"], width=2, dash="dot"),
            legendgroup="over",
            showlegend=(i == 0),
        ))

    # ── Group 4: Warm-start sample markers (from sampling_plans) ────────────
    for pi, plan in enumerate(plans):
        env = next(e for e in hist if abs(e.time - plan.time) < 1e-6)
        for ii, (interval_idx, sx_arr) in enumerate(plan.sample_indices.items()):
            sT = np.interp(sx_arr, env.x, env.T)
            fig.add_trace(go.Scatter3d(
                x=sx_arr, y=[plan.time]*len(sx_arr), z=sT,
                mode="markers",
                name=f"Warm-start (t={plan.time:.0f}s)",
                marker=dict(size=4, color=colors["samples"],
                            symbol="circle",
                            line=dict(color="darkred", width=0.5)),
                legendgroup="warm",
                showlegend=(pi == 0 and ii == 0),
            ))

    # ── Group 5: Refinement markers (Plasma, coloured by step order) ─────────
    n_warm = len(X_warm)
    X_ref  = X_train[n_warm:]
    T_ref  = T_train[n_warm:]
    if len(X_ref) > 0:
        steps = np.arange(len(X_ref), dtype=float)
        fig.add_trace(go.Scatter3d(
            x=X_ref[:,0], y=X_ref[:,1], z=T_ref,
            mode="markers",
            name=f"Refined ({len(X_ref)} pts, {strategy})",
#             marker=dict(
#                 size=5,
#                 color=steps,
#                 colorscale="blues",
#                 colorbar=dict(title="Step", thickness=12, x=1.04,
#                               tickfont=dict(size=10)),
#                 opacity=0.92,
#                 symbol="diamond",
#                 line=dict(color="black", width=0.5),
            marker=dict(size=4, color='green',
                            symbol="circle",
                            line=dict(color="green", width=0.5)
            ),
            hovertemplate=(
                "<b>Refined point</b><br>"
                "x=%{x:.4f} m<br>t=%{y:.1f} s<br>"
                "T=%{z:.2f} °C<br>step=%{marker.color:.0f}<extra></extra>"
            ),
        ))

    # ── Group 6: Source planes ────────────────────────────────────────────────
    if src_x:
        t_lo  = float(hist[0].time)
        t_hi  = float(hist[-1].time)
        T_min = float(T_train.min()) - 2.
        T_max = float(T_train.max()) + 5.
        for sx in src_x:
            fig.add_trace(go.Mesh3d(
                x=[sx, sx, sx, sx],
                y=[t_lo, t_lo, t_hi, t_hi],
                z=[T_min, T_max, T_max, T_min],
                opacity=0.12, color="red",
                name=f"Source x={sx:.3f} m",
                legendgroup="sources",
                showlegend=True,
                hoverinfo="skip",
            ))

    # ── Layout ────────────────────────────────────────────────────────────────
    fig.update_layout(
        title=dict(text=title, font=dict(size=14)),
        scene=dict(
            xaxis_title="Position x (m)",
            yaxis_title="Time t (s)",
            zaxis_title="Temperature T (°C)",
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.3)),
            aspectmode="manual",
            aspectratio=dict(x=1.6, y=2.2, z=1.2),
        ),
        legend=dict(x=0.01, y=0.99, font=dict(size=10), tracegroupgap=4),
        width=1200, height=800,
        margin=dict(l=0, r=60, b=0, t=50),
    )
    return fig


## C4. `run_uncertainty_exp`

UNCERT_GRID_NX=100
UNCERT_GRID_NT=100
UNCERT_N_REFINE=50
UNCERT_XI=0.0
UNCERT_REFIT_K=1
UNCERT_DUP_TOL=0.01
UNCERT_VIS_EVERY=15

def run_uncertainty_exp(n_sources,ref_splits,ref_x,ref_t,ref_Thist,
                        strategies=None,n_refine=UNCERT_N_REFINE,
                        grid_nx=UNCERT_GRID_NX,grid_nt=UNCERT_GRID_NT,
                        xi=UNCERT_XI,refit_k=UNCERT_REFIT_K,vis_every=UNCERT_VIS_EVERY):
    if strategies is None: strategies=list(ACQUISITION_STRATEGIES.keys())
    sc=SOURCE_CONFIGS[n_sources]; src_x_loc=sc["locs"]; tag_base=f"{n_sources}-source uncertainty"; seed_everything(42)
    print(f"\n{'='*65}\n  {tag_base.upper()}\n{'='*65}")
    interp_fn=RegularGridInterpolator((ref_t,ref_x),ref_Thist,method="linear",bounds_error=False,fill_value=None)
    (Xvs,yvs)=ref_splits["data"]["val_seen"]; (Xvu,yvu)=ref_splits["data"]["val_unseen"]; (Xte,yte)=ref_splits["data"]["test_unseen"]
    gx=np.linspace(ref_x[0],ref_x[-1],grid_nx); gt=np.linspace(ref_t[0],ref_t[-1],grid_nt)
    GX,GT=np.meshgrid(gx,gt); X_grid=np.stack([GX.ravel(),GT.ravel()],1)
    diag=math.sqrt((ref_x[-1]-ref_x[0])**2+(ref_t[-1]-ref_t[0])**2); delta=UNCERT_DUP_TOL*diag

    # Phase 1: warm start
    print("\n── Phase 1: Adaptive warm start")
    cfg=RodConfigSrc(**EXP_CFG,source_locs=tuple(src_x_loc),source_temps=tuple(sc["temps"]))
    sim=RodSimulationSrc(cfg); sampler=AdaptiveSampler(cfg.t_end,ADAP_RI,ADAP_N_SI); mc_env=McCormickEnvelope()
    for t_rec in sampler.record_times:
        T=sim.advance_to(t_rec); sampler.record(mc_env.compute_full(sim.x,T,t_rec))
    sampler.allocate_global_budget(ADAP_BUDGET,ADAP_MIN_S)
    ws_x,ws_t,ws_T=[],[],[]
    for plan in sampler.sampling_plans:
        env=next(e for e in sampler.envelope_history if abs(e.time-plan.time)<1e-6); seen=set()
        for k,sx_arr in plan.sample_indices.items():
            for xv in sx_arr:
                key=round(xv,10)
                if key not in seen:
                    seen.add(key); ws_x.append(float(xv)); ws_t.append(float(plan.time)); ws_T.append(float(np.interp(xv,env.x,env.T)))
    X_warm=np.stack([ws_x,ws_t],1); T_warm=np.array(ws_T); n_warm=len(T_warm)
    print(f"  Warm-start pts: {n_warm:,}")

    # Phase 2: gap weight
    print("\n── Phase 2: gap_weight"); gap_weight=build_gap_weight_grid(X_grid,sampler)
    plot_gap_weight_map(X_grid,gap_weight,grid_nx,grid_nt,src_x_loc,f"McCormick gap_weight — {n_sources} sources")

    # Phase 3: sequential refinement
    results={}
    for strategy in strategies:
        acq_fn=ACQUISITION_STRATEGIES[strategy]; tag=f"{tag_base} [{strategy}]"
        print(f"\n{'─'*60}\n  Strategy: {strategy}\n{'─'*60}"); seed_everything(42)
        X_tr=X_warm.copy(); T_tr=T_warm.copy(); scaler_u=StandardScaler().fit(X_tr); gpr_u=None; history=[]
        for step in range(n_refine):
            if step%refit_k==0 or gpr_u is None: gpr_u=train_gpr(scaler_u.transform(X_tr),T_tr,GPR_MAX)
            mu,std=gpr_u.predict(scaler_u.transform(X_grid),return_std=True); std=np.maximum(std,1e-12)
            T_best=float(T_tr.max()); acq=acq_fn(mu,std,gap_weight,T_best,xi)
            acq_work=acq.copy(); x_next=None
            for _ in range(20):
                idx=int(np.argmax(acq_work)); cand=X_grid[idx:idx+1]
                if np.linalg.norm(X_tr-cand,axis=1).min()>=delta: x_next=cand; break
                acq_work[idx]=-np.inf
            if x_next is None: x_next=X_grid[[np.random.randint(len(X_grid))]]
            T_next=float(interp_fn(x_next[:,[1,0]])[0]); X_tr=np.vstack([X_tr,x_next]); T_tr=np.append(T_tr,T_next); scaler_u=StandardScaler().fit(X_tr)
            if vis_every>0 and (step%vis_every==0 or step==n_refine-1):
                plot_acquisition_step(X_grid,acq,std,X_tr,x_next,grid_nx,grid_nt,ref_x,ref_t,ref_Thist,step,strategy,src_x=src_x_loc)
            if step%max(1,n_refine//10)==0 or step==n_refine-1:
                gpr_snap=train_gpr(scaler_u.transform(X_tr),T_tr,GPR_MAX); yp_te,_=gpr_snap.predict(scaler_u.transform(Xte),return_std=True)
                snap_rmse=math.sqrt(mean_squared_error(yte,yp_te))
                history.append(dict(step=step,test_rmse=snap_rmse,n_train=len(T_tr),x_next=x_next.ravel().tolist()))
                print(f"  step {step:4d}  n={len(T_tr):5d}  RMSE={snap_rmse:.4f}  x=({x_next[0,0]:.4f}m,{x_next[0,1]:.1f}s)")

        # Final surrogates
        gpr_final=train_gpr(scaler_u.transform(X_tr),T_tr,GPR_MAX); gpr_fn_u=gpr_predict_fn(gpr_final,scaler_u)
        Xvs_s=scaler_u.transform(Xvs); Xvu_s=scaler_u.transform(Xvu)
        mlp_u,hist_u=train_mlp(scaler_u.transform(X_tr),T_tr,Xvs_s,yvs,Xvu_s,yvu,**MLP_KW)
        mlp_u.eval(); mlp_fn_u=mlp_predict_fn(mlp_u,scaler_u); plot_train_curves(hist_u,f"MLP — {tag}")
        metrics={}
        for sn,X,y in [("val_seen",Xvs,yvs),("val_unseen",Xvu,yvu),("test_unseen",Xte,yte)]:
            yh_m,_=mlp_fn_u(X); yh_g,_=gpr_fn_u(X); metrics[f"MLP/{sn}"]=regression_metrics(y,yh_m); metrics[f"GPR/{sn}"]=regression_metrics(y,yh_g)
            print(f"  MLP {sn:12s}  RMSE={metrics[f'MLP/{sn}']['RMSE']:.4f}"); print(f"  GPR {sn:12s}  RMSE={metrics[f'GPR/{sn}']['RMSE']:.4f}")
        Xx,Xt=np.meshgrid(ref_x,ref_t); Xfull=np.stack([Xx.ravel(),Xt.ravel()],1)
        mf,_=mlp_fn_u(Xfull); gf,gs=gpr_fn_u(Xfull)
        mg=mf.reshape(len(ref_t),len(ref_x)); gg=gf.reshape(len(ref_t),len(ref_x)); sg=gs.reshape(len(ref_t),len(ref_x))
        fig,axes=plt.subplots(2,3,figsize=(18,8))
        for ax,(Z,ttl) in zip(axes.flat,[(mg,"MLP pred"),(np.abs(mg-ref_Thist),"MLP |error|"),(gg,"GPR pred"),(np.abs(gg-ref_Thist),"GPR |error|"),(sg,"GPR σ"),(ref_Thist,"Truth")]):
            plot_field_src(ref_x,ref_t,Z,ttl,src_x=src_x_loc,ax=ax)
        fig.suptitle(tag,fontsize=12); plt.tight_layout(); plt.show()
        plot_profiles_src(ref_x,ref_t,ref_Thist,mlp_fn_u,gpr_fn_u,PROFILE_TIMES,src_x_loc,f"Profiles — {tag}")

        # 3-D interactive scatter (updated style — sampler passed in)
        plot_uncertainty_3d(
            X_train  = X_tr,
            T_train  = T_tr,
            X_warm   = X_warm,
            strategy = strategy,
            sampler  = sampler,
            src_x    = src_x_loc,
            title    = f"Uncertainty 3-D — {tag}  ({len(T_tr)} pts)",
        ).show()

        results[strategy]=dict(X_train=X_tr,y_train=T_tr,X_warm=X_warm,scaler=scaler_u,
            gpr=gpr_final,gpr_fn=gpr_fn_u,mlp=mlp_u,mlp_fn=mlp_fn_u,
            metrics=metrics,n_train=len(T_tr),history=history,sampler=sampler)

    plot_rmse_convergence({s:results[s]["history"] for s in strategies},f"Test RMSE convergence — {n_sources} sources")
    return results,X_warm,n_warm,gap_weight,X_grid,grid_nx,grid_nt



uniform_all={}; adaptive_all={}; bisect_all={}; uncertainty_all={}; _uncert_meta={}

for n_src in [2,3,4]:
    print(f"\n{'#'*70}\n##  n_sources = {n_src}\n{'#'*70}")
    uniform_all[n_src]=run_uniform_exp(n_src)
    adaptive_all[n_src]=run_adaptive_exp(n_src,ref_splits=uniform_all[n_src]["splits"],ref_x=uniform_all[n_src]["x"],ref_t=uniform_all[n_src]["t"],ref_Thist=uniform_all[n_src]["Thist"])
    bisect_all[n_src]=run_bisection_exp(n_src,ref_splits=uniform_all[n_src]["splits"],ref_x=uniform_all[n_src]["x"],ref_t=uniform_all[n_src]["t"],ref_Thist=uniform_all[n_src]["Thist"])
    res_u,X_warm,n_warm,gw,Xg,gnx,gnt=run_uncertainty_exp(n_src,ref_splits=uniform_all[n_src]["splits"],ref_x=uniform_all[n_src]["x"],ref_t=uniform_all[n_src]["t"],ref_Thist=uniform_all[n_src]["Thist"])
    uncertainty_all[n_src]=res_u; _uncert_meta[n_src]=dict(X_warm=X_warm,n_warm=n_warm,gap_weight=gw,X_grid=Xg,grid_nx=gnx,grid_nt=gnt)

print("\n✓ All experiments complete.")


## C6. Post-run Visualisation Suite

# C6a — Acquisition snapshot + sample trajectories
for n_src in [2,3,4]:
    meta=_uncert_meta[n_src]; res_u=uncertainty_all[n_src]; src_x=SOURCE_CONFIGS[n_src]["locs"]
    ref_x=uniform_all[n_src]["x"]; ref_t=uniform_all[n_src]["t"]
    X_warm=meta["X_warm"]; gw=meta["gap_weight"]; X_grid=meta["X_grid"]; gnx=meta["grid_nx"]; gnt=meta["grid_nt"]
    scaler_ws=StandardScaler().fit(X_warm); T_warm=res_u["STD"]["y_train"][:meta["n_warm"]]
    gpr_ws=train_gpr(scaler_ws.transform(X_warm),T_warm,GPR_MAX)
    mu0,std0=gpr_ws.predict(scaler_ws.transform(X_grid),return_std=True); std0=np.maximum(std0,1e-12); T_best0=float(T_warm.max())
    xn_dict={}
    for s in res_u:
        hist=res_u[s]["history"]; xn=hist[0]["x_next"] if hist else None
        xn_dict[s]=np.array(xn).reshape(1,2) if xn is not None else None
    plot_all_acquisitions_snapshot(mu0,std0,gw,T_best0,X_grid,gnx,gnt,xn_dict,src_x=src_x,step=0,title=f"Acquisition comparison — {n_src} sources")
    plot_sample_trajectory(res_u,n_warm=meta["n_warm"],ref_x=ref_x,ref_t=ref_t,src_x=src_x,title=f"Sample trajectory — {n_src} sources")


# C6b — Sampling density all methods
for n_src in [2,3,4]:
    res_u=uncertainty_all[n_src]; src_x=SOURCE_CONFIGS[n_src]["locs"]
    adap_res=adaptive_all[n_src]; a_xs,a_ts=[],[]
    for plan in adap_res["sampler"].sampling_plans:
        for k,sx_arr in plan.sample_indices.items(): a_xs.extend(sx_arr.tolist()); a_ts.extend([plan.time]*len(sx_arr))
    sxt_adap=np.stack([a_xs,a_ts],1); Xtr_unif=uniform_all[n_src]["splits"]["data"]["train"][0]
    plot_full_density_comparison(uncert_results=res_u,bisect_X=bisect_all[n_src]["X_train"],adapt_X=sxt_adap,unif_X=Xtr_unif,src_x=src_x,title=f"Sampling density — all methods — {n_src} sources")


# C6c — GPR field per strategy
for n_src in [2,3,4]:
    plot_strategy_error_fields(uncertainty_all[n_src],uniform_all[n_src]["x"],uniform_all[n_src]["t"],uniform_all[n_src]["Thist"],SOURCE_CONFIGS[n_src]["locs"],f"GPR field — per strategy — {n_src} sources")


# C6d — Profile comparison
for n_src in [2,3,4]:
    plot_strategy_profiles(uncertainty_all[n_src],uniform_all[n_src]["x"],uniform_all[n_src]["t"],uniform_all[n_src]["Thist"],PROFILE_TIMES,SOURCE_CONFIGS[n_src]["locs"],f"T(x) profiles — all strategies — {n_src} sources")


# C6e — Pred vs true
for n_src in [2,3,4]:
    Xte,yte=uniform_all[n_src]["splits"]["data"]["test_unseen"]
    plot_strategy_pred_vs_true(uncertainty_all[n_src],Xte,yte,f"GPR pred vs true — {n_src} sources")


ModuleNotFoundError: No module named 'seaborn'